# Phân loại cảm xúc IMDb — basic
Phương pháp: **Count unigram khoảng trắng + MultinomialNB**, học toàn bộ đặc trưng và tham số từ train, không dùng pretrained.
Nguồn: [imdb_reviews](https://www.tensorflow.org/datasets/catalog/imdb_reviews). Đầu vào là một văn bản, đầu ra là nhãn trong `LABEL_NAMES`.
Bảng người dùng: dòng 2, metric chính Macro F1 tăng; Accuracy bổ sung.
Chạy độc lập từ kernel mới. Thư mục làm việc là nơi lưu `data/` và `artifacts/`; sửa hai đường dẫn nếu cần.

**Bản basic cập nhật:** Unigram counts từ khoảng trắng thay TF-IDF; raw model input không normalize HTML/NFKC; giữ nguyên canonical grouping để raw split không đổi và giữ MultinomialNB. Không phải bản tối ưu; không cam kết chênh lệch điểm.

**Cách viết training/checkpoint:** training loop được viết trực tiếp trong cell (sklearn dùng fit); lưu gọn khi validation cải thiện. Checkpoint phục vụ inference, không lưu optimizer/scheduler/RNG để resume.


## 1. Môi trường và cấu hình cố định
Dùng Python 3.11/3.12 trong Jupyter/Colab. Nếu thiếu thư viện, chạy trong một ô riêng:
`%pip install numpy scipy scikit-learn joblib torch`.
Riêng IMDb tải online qua TFDS cần `%pip install tensorflow tensorflow-datasets`;
archive Stanford gốc cục bộ không cần TensorFlow. Dùng PyTorch đang có, không tự thay wheel CUDA.
Mô hình thưa dùng CPU: batch/epoch/LR của mạng neural không áp dụng; cấu hình cố định là
số đặc trưng, C/alpha, ngưỡng hội tụ và số vòng lặp solver. GPU 32 GB là mục tiêu chung,
bài này không cần GPU để phù hợp biểu diễn thưa và kích thước dữ liệu.


In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import html
import importlib.metadata
import json
import random
import re
import unicodedata
import numpy as np
import torch

SEED = 42
PRETRAINING_POLICY = "scratch"
PRETRAINED_COMPONENTS = []
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def sha(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def fingerprint(rows):
    return sha(json.dumps(rows, ensure_ascii=False, sort_keys=True, separators=(",", ":")))

def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    if not rows:
        raise ValueError(f"Tệp dữ liệu rỗng: {path}")
    return rows

def package_versions(names):
    result = {}
    for name in names:
        try:
            result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            result[name] = "chưa cài"
    return result
import joblib
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.exceptions import ConvergenceWarning
import warnings

TASK = 'imdb_reviews'
VARIANT = 'basic'
SOURCE = 'https://www.tensorflow.org/datasets/catalog/imdb_reviews'
LABEL_NAMES = ['neg', 'pos']
DATA_DIR = Path("data/sentiment_imdb_reviews")
OUTPUT_DIR = Path("artifacts/sentiment_imdb_reviews/basic")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
WORD_MAX_FEATURES = 40000
CHAR_MAX_FEATURES = 100000
MIN_DF = 1
C_VALUES = (0.25, 1.0, 4.0)
NB_ALPHA = 1.0
NB_RATIO_ALPHA = 1.0
MAX_ITER = 10000
TOLERANCE = 1e-4
USE_NB_RATIO = False
PRIMARY_METRIC = "macro_f1"
VERSIONS = package_versions(["numpy", "scipy", "scikit-learn", "joblib", "torch", "tensorflow-datasets"])
print("Phương pháp: Count unigram khoảng trắng + MultinomialNB; thiết bị khả dụng:", DEVICE)
print("Đặc trưng thưa và mô hình sklearn chạy CPU, không cần GPU.")
print("Phiên bản thư viện:", VERSIONS)


## 2. Đọc dữ liệu, chuẩn hóa và chia validation
Ưu tiên `DATA_DIR/train.jsonl` và `test.jsonl` đúng bộ dữ liệu, mỗi dòng có `text: str`,
`label: int`; không tự thay nguồn khi lỗi. IMDb còn hỗ trợ `aclImdb_v1.tar.gz` chính gốc.
20 Newsgroups JSONL phải đã loại headers/footers/quotes bằng loader sklearn.
Giữ official test để đánh giá cuối; chọn fold 0 của StratifiedGroupKFold 5 fold trên train.
Nhóm bản sao theo văn bản chuẩn hóa không phân biệt hoa/thường; tỷ lệ validation xấp xỉ 20%.
Không xóa stopword/phủ định; không bỏ mẫu rỗng. In dấu vân tay để đối chiếu hai biến thể.


In [ ]:
def normalize_text(text):
    text = html.unescape(unicodedata.normalize("NFKC", text))
    text = re.sub(r"<br\s*/?>", " ", text, flags=re.I)
    return " ".join(text.split())

def canonical_rows(rows, split):
    result = []
    for row in rows:
        if not isinstance(row.get("text"), str):
            raise ValueError("Schema cần text dạng chuỗi và label số nguyên.")
        y = row.get("label")
        if isinstance(y, bool) or not isinstance(y, int) or not 0 <= y < len(LABEL_NAMES):
            raise ValueError(f"Nhãn sai: {y!r}")
        text = row["text"]
        result.append({"text": text, "label": y, "key": sha(text + "\0" + str(y))})
    # Sắp theo nội dung để TFDS/JSONL/tar cùng nguồn có thứ tự chia giống nhau.
    result.sort(key=lambda x: (x["key"], x["text"]))
    return [{**r, "id": f"{split}:{i}:{r['key']}"} for i, r in enumerate(result)]

def load_raw(split):
    local_path = DATA_DIR / f"{split}.jsonl"
    if local_path.exists():
        print(f"Đọc dữ liệu gốc cục bộ: {local_path}")
        return canonical_rows(read_jsonl(local_path), split)
    if TASK == "imdb_reviews":
        archive = DATA_DIR / "aclImdb_v1.tar.gz"
        if archive.exists():
            import tarfile
            rows = []
            with tarfile.open(archive, "r:gz") as tar:
                for member in tar:
                    parts = member.name.split("/")
                    if (member.isfile() and len(parts) == 4 and parts[0] == "aclImdb"
                            and parts[1] == split and parts[2] in ("neg", "pos")
                            and parts[3].endswith(".txt")):
                        with tar.extractfile(member) as f:
                            # Cùng quy tắc với builder TFDS: đọc trọn review, strip hai đầu.
                            text = f.read().decode("utf-8").strip()
                        rows.append({"text": text, "label": int(parts[2] == "pos")})
            if not rows:
                raise ValueError(f"Không tìm thấy aclImdb/{split}/neg|pos/*.txt trong {archive}")
            print(f"Đọc archive Stanford gốc, cùng nguồn TFDS: {archive}")
            return canonical_rows(rows, split)
        try:
            import tensorflow as tf
            tf.config.set_visible_devices([], "GPU")  # Chỉ dùng TensorFlow để đọc dữ liệu CPU.
            import tensorflow_datasets as tfds
        except ImportError as exc:
            raise RuntimeError("Cài tensorflow tensorflow-datasets trong Python được hỗ trợ; "
                "hoặc đặt archive Stanford aclImdb_v1.tar.gz trong DATA_DIR. "
                "Có thể dùng train.jsonl/test.jsonl export đúng IMDb, text:str, label:0|1.") from exc
        try:
            ds, info = tfds.load("imdb_reviews/plain_text:1.0.0", split=split,
                as_supervised=True, with_info=True, shuffle_files=False,
                data_dir=str(DATA_DIR / "tfds"))
            assert info.features["label"].names == ["neg", "pos"]
            rows = [{"text": t.decode("utf-8"), "label": int(y)} for t,y in tfds.as_numpy(ds)]
        except Exception as exc:
            raise RuntimeError("Không tải/đọc được IMDb từ TFDS. Kiểm tra mạng, thư mục cache "
                "hoặc đặt archive Stanford gốc trong DATA_DIR; không đổi dataset.") from exc
    else:
        from sklearn.datasets import fetch_20newsgroups
        try:
            data = fetch_20newsgroups(subset=split, remove=("headers", "footers", "quotes"),
                shuffle=False, data_home=str(DATA_DIR / "sklearn"), download_if_missing=True)
        except Exception as exc:
            raise RuntimeError("Không tải/đọc được 20 Newsgroups. Kiểm tra mạng/cache; "
                "hoặc cung cấp train.jsonl/test.jsonl đã remove headers, footers, quotes, "
                "nhãn theo LABEL_NAMES. Không dùng fetch_20newsgroups_vectorized.") from exc
        assert list(data.target_names) == LABEL_NAMES, "Thứ tự nhãn khác khai báo."
        rows = [{"text": text, "label": int(y)} for text,y in zip(data.data, data.target)]
    return canonical_rows(rows, split)

raw_train = load_raw("train")
texts_all = [normalize_text(r["text"]) for r in raw_train]
y_all = np.array([r["label"] for r in raw_train], dtype=np.int64)
# Bản sao văn bản ở cùng fold; văn bản rỗng vẫn là mẫu hợp lệ, nhóm riêng từng id.
groups = np.array([sha(t.casefold()) if t else sha(r["id"])
                   for r,t in zip(raw_train, texts_all)])
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, val_idx = next(splitter.split(texts_all, y_all, groups))
assert not (set(groups[train_idx]) & set(groups[val_idx]))
assert set(y_all[train_idx]) == set(range(len(LABEL_NAMES)))
assert set(y_all[val_idx]) == set(range(len(LABEL_NAMES)))
train_text = [raw_train[i]["text"] for i in train_idx]
val_text = [raw_train[i]["text"] for i in val_idx]
y_train, y_val = y_all[train_idx], y_all[val_idx]
SPLIT_INFO = {
    "source": SOURCE, "seed": SEED, "splitter": "StratifiedGroupKFold, fold 0/5",
    "train": fingerprint([raw_train[i] for i in train_idx]),
    "validation": fingerprint([raw_train[i] for i in val_idx]),
    "counts": {"raw_train": len(raw_train), "train": len(train_idx), "validation": len(val_idx)},
}
print("Số mẫu và dấu vân tay split:", json.dumps(SPLIT_INFO, ensure_ascii=False, indent=2))
print("Phân bố nhãn train:", np.bincount(y_train, minlength=len(LABEL_NAMES)).tolist())
print("Phân bố nhãn validation:", np.bincount(y_val, minlength=len(LABEL_NAMES)).tolist())
print("Số văn bản rỗng sau chuẩn hóa:", sum(not t for t in texts_all))
print("Số bản sao văn bản không rỗng:", len(groups) - len(set(groups)))
print("Ví dụ train:", train_text[0][:250], "| nhãn:", LABEL_NAMES[int(y_train[0])])


## 3. Biểu diễn văn bản chỉ học trên train
Từ vựng, IDF và log-ratio nhãn (nếu dùng) chỉ được fit sau khi chia train/validation.
Validation/test chỉ transform. Mọi ma trận vẫn thưa; không chuyển toàn bộ sang dense.


In [ ]:
def build_features():
    return CountVectorizer(tokenizer=str.split, token_pattern=None,
        lowercase=False, min_df=1, max_features=WORD_MAX_FEATURES, dtype=np.int64)

features = build_features()
X_train = features.fit_transform(train_text)
X_val = features.transform(val_text)
ratio = np.ones(X_train.shape[1], dtype=np.float32)
print("Kích thước đặc trưng train/validation:", X_train.shape, X_val.shape)


## 4. Mô hình, huấn luyện và chọn checkpoint
Count unigram khoảng trắng + MultinomialNB. Alpha cố định 1.0; validation để kiểm tra pipeline.
Lưu toàn bộ pipeline với joblib; không refit trên validation sau lựa chọn.


In [ ]:
PARAMETERS = (NB_ALPHA,)
def make_classifier(parameter):
    return MultinomialNB(alpha=parameter)

def classification_metrics(y, pred):
    return {"macro_f1": float(f1_score(y, pred, labels=list(range(len(LABEL_NAMES))),
        average="macro", zero_division=0)), "accuracy": float(accuracy_score(y, pred))}

CHECKPOINT_PATH = OUTPUT_DIR / "best.joblib"
best_score = -float("inf")
history = []
for parameter in PARAMETERS:
    classifier = make_classifier(parameter)
    # Không chấp nhận nghiệm SVM chưa hội tụ rồi âm thầm báo kết quả.
    with warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        classifier.fit(X_train, y_train)
    val_pred = classifier.predict(X_val)
    scores = classification_metrics(y_val, val_pred)
    history.append({"parameter": float(parameter), **scores})
    print("Tham số:", parameter, "| validation:", scores)
    if scores[PRIMARY_METRIC] > best_score:
        best_score = scores[PRIMARY_METRIC]
        # Chỉ lưu các thành phần đã fit cần để inference; cấu hình/provenance ở notebook.
        joblib.dump({
            "features": features,
            "ratio": ratio,
            "classifier": classifier,
            "label_names": LABEL_NAMES,
            "validation": scores,
        }, CHECKPOINT_PATH)
(OUTPUT_DIR / "validation_history.json").write_text(
    json.dumps(history, ensure_ascii=False, indent=2), encoding="utf-8")
print("Đã lưu checkpoint tốt nhất theo Macro F1 validation:", CHECKPOINT_PATH)


## 5. Nạp lại, đánh giá test một lần và dự đoán
Test được đọc sau khi chốt mô hình. Macro F1 tính trên toàn bộ danh sách lớp,
`zero_division=0`, thang 0–1; Accuracy bổ sung. Thống kê trùng nguồn chỉ là chẩn đoán,
không dùng để đổi mô hình. Không lấy test làm tín hiệu lựa chọn C/alpha.


In [ ]:
# Chỉ nạp checkpoint do chính notebook này tạo; joblib không dành cho tệp không tin cậy.
best = joblib.load(CHECKPOINT_PATH)

def predict_texts(texts):
    normalized = [str(t) for t in texts]
    x = best["features"].transform(normalized).multiply(best["ratio"]).tocsr()
    return best["classifier"].predict(x)

# Đối chiếu trước khi chạm test, không fit lại trên train+validation.
reloaded_val = predict_texts([raw_train[i]["text"] for i in val_idx])
assert abs(classification_metrics(y_val, reloaded_val)[PRIMARY_METRIC]
           - best["validation"][PRIMARY_METRIC]) < 1e-12
raw_test = load_raw("test")
test_text = [r["text"] for r in raw_test]
y_test = np.array([r["label"] for r in raw_test])
test_fingerprint = fingerprint(raw_test)
test_groups = {sha(normalize_text(t).casefold()) for t in test_text if normalize_text(t)}
overlap = len(test_groups & {sha(t.casefold()) for t in texts_all if t})
print("Dấu vân tay test:", test_fingerprint)
print("Số văn bản không rỗng trùng giữa official train và test:", overlap)
if overlap:
    print("Có trùng dữ liệu nguồn: giữ split chuẩn để so sánh, diễn giải test thận trọng.")
test_pred = predict_texts(test_text)
test_scores = classification_metrics(y_test, test_pred)
print("Kết quả test của pipeline đã chốt:", test_scores)
print(classification_report(y_test, test_pred, labels=list(range(len(LABEL_NAMES))),
    target_names=LABEL_NAMES, zero_division=0, digits=4))
print("Ma trận nhầm lẫn (hàng=nhãn thật, cột=dự đoán):")
print(confusion_matrix(y_test, test_pred, labels=list(range(len(LABEL_NAMES)))))
(OUTPUT_DIR / "test_metrics.json").write_text(json.dumps({
    "validation": best["validation"], "test": test_scores,
    "test_fingerprint": test_fingerprint, "source_overlap": overlap,
    "split_info": SPLIT_INFO, "versions": VERSIONS}, ensure_ascii=False, indent=2), encoding="utf-8")
with (OUTPUT_DIR / "test_predictions.jsonl").open("w", encoding="utf-8") as f:
    for row, pred in zip(raw_test, test_pred):
        f.write(json.dumps({"id": row["id"], "label": row["label"], "prediction": int(pred)},
            ensure_ascii=False) + "\n")
print("Ví dụ dự đoán:", LABEL_NAMES[int(predict_texts([test_text[0]])[0])])


## 6. Thực hành nâng cấp
Bắt đầu từ bản basic: thêm bigram vào **Biểu diễn văn bản**, giữ nguyên split,
so sánh Macro F1 validation. Xem lộ trình chi tiết và trạng thái kiểm tra trong
`huong_giai_quyet.md`. Không có chênh lệch điểm được bảo đảm trước khi đo.
